### 1. Environment Initialization
Importing necessary dependencies for audio signal processing, deep learning operations, and data manipulation. System warnings are suppressed to maintain standard output clarity during iterative execution.

In [ ]:
import os
import random
import warnings
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm
import timm
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR

warnings.filterwarnings('ignore')
print(" Imports loaded successfully!")

### 2. Configuration and Dynamic Pathing
Defining global hyperparameters and architectural configurations. The audio sampling rate ($f_s$) is strictly set to 32 kHz with a fixed temporal context window of 5 seconds. Parameters for Per-Channel Energy Normalization (PCEN)—specifically $\alpha, \delta, r,$ and $s$—are declared here to optimize the dynamic foreground-background acoustic separation.

In [ ]:
def find_path(name, extension=None):
    """Finds files or directories dynamically to avoid hardcoded Kaggle paths."""
    for root, dirs, files in os.walk('/kaggle/input'):
        if extension:
            for f in files:
                if name in f and f.endswith(extension):
                    return Path(root) / f
        else:
            if name in dirs or name in files:
                return Path(root) / name
    return None

class CFG:
    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Audio Specs (5-second windows)
    sr = 32000
    duration = 5 
    n_mels = 128
    fmin = 40
    fmax = 15000
    
    # PCEN Parameters (normalizes background noise)
    pcen_alpha = 0.98
    pcen_delta = 2.0
    pcen_r = 0.5
    pcen_s = 0.025 
    
    # Dual Training Hyperparameters
    batch_size = 32
    pure_epochs = 5        # Phase 1: Overfit pure audio
    pure_lr = 1e-4
    forest_epochs = 15     # Phase 2: Generalize on noisy IRL
    forest_lr = 2e-5       # Gentle fine-tuning LR
    
    BASE_DIR = find_path("sample_submission.csv").parent if find_path("sample_submission.csv") else Path('.')
    TRAIN_AUDIO_DIR = find_path("train_audio")
    SAMPLE_SUB = find_path("sample_submission.csv")

print(f"Running on: {CFG.device}")

### 3. Dataset Generation: Clean and Noisy Domains
This section constructs the dual-domain training metadata. The primary dataset consists of clean, localized vocalizations mapped to a canonical 234-class target space. A secondary dataset is sampled to represent complex, multi-label real-world soundscapes, which will be utilized to force model generalization in later epochs.

In [ ]:
# 1. Get official 234 species
sample_sub = pd.read_csv(CFG.SAMPLE_SUB) if CFG.SAMPLE_SUB else pd.DataFrame(columns=['row_id'] + [f'bird_{i}' for i in range(234)])
species_cols = list(sample_sub.columns[1:])
label_to_id = {label: i for i, label in enumerate(species_cols)}

# 2. Crawl pure audio 
pure_data = []
if CFG.TRAIN_AUDIO_DIR:
    found_species = [d for d in os.listdir(CFG.TRAIN_AUDIO_DIR) if d in label_to_id]
    for species in tqdm(found_species, desc="Crawling Pure Audio"):
        species_path = CFG.TRAIN_AUDIO_DIR / species
        for audio_file in os.listdir(species_path):
            if audio_file.endswith(".ogg"):
                pure_data.append({
                    "primary_label": species,
                    "path": str(species_path / audio_file)
                })

pure_df = pd.DataFrame(pure_data)

# 3. Simulate Forest/Noisy IRL data
# (Swap this with your actual noisy forest dataset path)
forest_df = pure_df.sample(min(1000, len(pure_df))) 
forest_df['start'] = '0:00' 

print(f" Found {len(pure_df)} pure audio files.")
print(f" Found {len(forest_df)} noisy IRL segments.")

### 4. Per-Channel Energy Normalization (PCEN)
Standard Mel-spectrograms suffer from variance in ambient background noise. We apply PCEN, which normalizes the spectrogram $E(t, f)$ using a temporally smoothed version $M(t, f)$. The operation is mathematically defined as:

$$PCEN(t, f) = \left( \frac{E(t, f)}{(M(t, f) + \epsilon)^\alpha} + \delta \right)^r - \delta^r$$

This formulation acts as a dynamic range compressor. It aggressively whitens stationary background noise while enhancing the transient, high-energy acoustic signatures typical of avian vocalizations.

In [ ]:
def compute_pcen(audio_path, offset=0.0):
    """
    Per-Channel Energy Normalization (PCEN).
    Whites out constant background noise so bird calls pop.
    """
    try:
        # Load exactly 5s chunk of audio
        y, _ = librosa.load(audio_path, sr=CFG.sr, offset=offset, duration=CFG.duration)
        
        # Pad with zeros if short
        if len(y) < CFG.sr * CFG.duration:
            y = np.pad(y, (0, CFG.sr * CFG.duration - len(y)))

        # 1. Magnitude Spectrogram
        S = librosa.feature.melspectrogram(
            y=y, sr=CFG.sr, n_mels=CFG.n_mels, 
            fmin=CFG.fmin, fmax=CFG.fmax, power=1
        )

        # 2. Apply PCEN 
        spec = librosa.pcen(
            S * (2**31), sr=CFG.sr, gain=CFG.pcen_s, 
            bias=CFG.pcen_delta, power=CFG.pcen_r, b=CFG.pcen_alpha
        )

        # 3. Normalize to 0-255 for image representation
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)
        return (spec * 255).astype(np.float32)
    
    except Exception as e:
        return np.zeros((CFG.n_mels, int(CFG.sr * CFG.duration / 512) + 1), dtype=np.float32)

### 5. Acoustic Feature Representation Analysis
A comparative visualization of the standard logarithmic Mel-spectrogram (dB scale) versus the PCEN-transformed representation. This empirical check validates the efficacy of PCEN in suppressing stationary acoustic noise prior to feature extraction.

In [ ]:
if len(pure_df) > 0:
    sample = pure_df.sample(1).iloc[0]
    
    y, _ = librosa.load(sample['path'], sr=CFG.sr, duration=CFG.duration)
    S_raw = librosa.feature.melspectrogram(y=y, sr=CFG.sr, n_mels=CFG.n_mels)
    S_db = librosa.power_to_db(S_raw, ref=np.max)
    
    S_pcen = compute_pcen(sample['path'])

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    img1 = axes[0].imshow(S_db, aspect='auto', origin='lower', cmap='magma')
    axes[0].set_title(f"Standard Mel-Spectrogram - {sample['primary_label']}")
    fig.colorbar(img1, ax=axes[0], format="%+2.0f dB")
    
    img2 = axes[1].imshow(S_pcen, aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title(f"PCEN Representation - {sample['primary_label']}")
    fig.colorbar(img2, ax=axes[1])
    
    plt.tight_layout()
    plt.show()

### 6. Constructing the Dataset Loaders
Two specialized `Dataset` classes are instantiated to handle our distinct training phases:
1. `PureAudioDataset` applies stochastic temporal cropping to strictly isolate transient signals. This is critical to prevent the network from overfitting to ambient silence at the beginning of clean recordings. 
2. `ForestNoisyDataset` maps exact temporal offsets and employs multi-label one-hot encoding ($y \in \{0, 1\}^{234}$) to compute loss against overlapping vocalizations in complex soundscapes.

In [ ]:
class PureAudioDataset(Dataset):
    """Dataset for Phase 1: Clean, focal bird audio to learn pure patterns."""
    def __init__(self, df, label_to_id):
        self.df = df
        self.label_to_id = label_to_id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Random crop to ensure we don't overfit to silence at the start of files
        try:
            duration = librosa.get_duration(path=row['path'])
            max_offset = max(0.0, duration - CFG.duration)
            random_offset = random.uniform(0.0, max_offset)
        except:
            random_offset = 0.0

        spec = compute_pcen(row['path'], offset=random_offset)
        image = np.stack([spec, spec, spec], axis=0)
        
        label = np.zeros(len(self.label_to_id), dtype=np.float32)
        if row['primary_label'] in self.label_to_id:
            label[self.label_to_id[row['primary_label']]] = 1.0
        
        return torch.tensor(image).float() / 255.0, torch.tensor(label).float()

class ForestNoisyDataset(Dataset):
    """Dataset for Phase 2: IRL noisy soundscapes to generalize the model."""
    def __init__(self, df, label_to_id):
        self.df = df
        self.label_to_id = label_to_id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Parse timestamp offset
        try:
            parts = str(row['start']).split(':')
            offset = sum(float(x) * 60 ** i for i, x in enumerate(reversed(parts)))
        except:
            offset = 0.0
            
        spec = compute_pcen(row['path'], offset=offset)
        image = np.stack([spec, spec, spec], axis=0)
        
        # Multi-label encoding
        label = np.zeros(len(self.label_to_id), dtype=np.float32)
        primary_labels = str(row['primary_label']).split(';')
        for bird in primary_labels:
            if bird in self.label_to_id:
                label[self.label_to_id[bird]] = 1.0
        
        return torch.tensor(image).float() / 255.0, torch.tensor(label).float()

print(" Dual datasets initialized.")

### 7. ConvNeXt Backbone Initialization
A pre-trained ConvNeXt-Tiny architecture is instantiated to extract spatial features from the pseudo-image representations (RGB-stacked spectrograms). The classification head is replaced with a linear projection matching our 234-dimensional target space, regularized via dropout to mitigate over-parameterization on the training set.

In [ ]:
class BirdConvNeXt(nn.Module):
    def __init__(self, model_name='convnext_tiny.fb_in1k', num_classes=234):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, in_chans=3)
        n_features = self.backbone.head.fc.in_features
        
        self.backbone.head.fc = nn.Sequential(
            nn.Dropout(0.2), 
            nn.Linear(n_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = BirdConvNeXt(num_classes=len(label_to_id)).to(CFG.device)
print(" Model Built: ConvNeXt Tiny")

### 8. Optimization Strategy: One-Cycle Policy
Visualizing the One-Cycle Learning Rate scheduler. This approach anneals the learning rate symmetrically, enabling rapid traversal of saddle points in the loss landscape. It results in a more robust local minimum convergence compared to standard step decay.

In [ ]:
dummy_optimizer = torch.optim.Adam(model.parameters(), lr=CFG.pure_lr)
dummy_scheduler = OneCycleLR(dummy_optimizer, max_lr=CFG.pure_lr * 5, total_steps=1000)

lrs = []
for _ in range(1000):
    dummy_optimizer.step()
    lrs.append(dummy_scheduler.get_last_lr()[0])
    dummy_scheduler.step()

plt.figure(figsize=(8, 4))
plt.plot(lrs, color='blue', linewidth=2)
plt.title("OneCycle Learning Rate Schedule", fontsize=14)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Learning Rate", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.fill_between(range(1000), lrs, color='blue', alpha=0.1)
plt.show()

### 9. Phase 1: Clean Domain Overfitting
The first optimization phase focuses exclusively on pristine audio data. Utilizing a relatively high maximum learning rate ($1 \times 10^{-4}$), the objective is to force the network to construct robust intermediate representations of isolated avian morphologies. The objective function is computed via Binary Cross-Entropy with Logits.

In [ ]:
pure_ds = PureAudioDataset(pure_df, label_to_id)
pure_loader = DataLoader(pure_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.pure_lr, weight_decay=1e-3)
criterion = nn.BCEWithLogitsLoss()
scheduler = OneCycleLR(optimizer, max_lr=CFG.pure_lr * 5, 
                       steps_per_epoch=len(pure_loader), epochs=CFG.pure_epochs)

print(f" Phase 1: Training on {len(pure_df)} pure samples...")
for epoch in range(CFG.pure_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(pure_loader, desc=f"Pure Epoch {epoch+1}/{CFG.pure_epochs}")
    
    for images, labels in pbar:
        images, labels = images.to(CFG.device), labels.to(CFG.device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    torch.save(model.state_dict(), f"convnext_pure_epoch{epoch+1}.pth")

print(" Phase 1 Complete!")

### 10. Phase 2: Real-World Generalization
The second optimization phase introduces the noisy, multi-label dataset. To preserve the delicate feature extractors established in Phase 1, the learning rate is scaled down by a factor of 5 ($2 \times 10^{-5}$). This phase forces the network to generalize its representations, identifying known acoustic patterns amidst significant real-world degradation.

In [ ]:
forest_ds = ForestNoisyDataset(forest_df, label_to_id)
forest_loader = DataLoader(forest_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.forest_lr, weight_decay=1e-3)
scheduler = OneCycleLR(optimizer, max_lr=CFG.forest_lr * 5, 
                       steps_per_epoch=len(forest_loader), epochs=CFG.forest_epochs)

print(f"Phase 2: Fine-tuning on {len(forest_df)} noisy forest segments...")
for epoch in range(CFG.forest_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(forest_loader, desc=f"Forest Epoch {epoch+1}/{CFG.forest_epochs}")
    
    for images, labels in pbar:
        images, labels = images.to(CFG.device), labels.to(CFG.device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    torch.save(model.state_dict(), f"convnext_final_v1_epoch{epoch+1}.pth")

print("Phase 2 Complete! Your model is now a Forest Expert.")